### ============================================================
### Finetune_transformer.ipynb
####  we fine-tune a pre-trained Transformer model (e.g., `roberta-base` or `bert-base-uncased`) for stance classification on the Gun Control dataset. The model learns to predict whether a tweet expresses *support* or *opposition* toward gun control.
### ============================================================


In [10]:
#Libraries 
import os
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset,load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch

In [11]:
# Paths
os.chdir("/workspace/dzuniga/multimodal-argmining")
DATA_PATH   = "data/"
IMG_PATH    = "data/images"
OUTPUT_DIR  = "results/text/finetune_transformer/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [12]:
#Load Dataset
df = pd.read_csv(os.path.join(DATA_PATH, "new_dataset_annotated.csv"))

df_train = df[df["split"]=="train"]
df_dev = df[df["split"]=="dev"]
df_test = df[df["split"]=="test"]

# Map labels
stance_2id = {"oppose": 0, "support": 1}
for df in [df_train, df_dev, df_test]:
    df["label"] = df["stance"].map(stance_2id)

print(f"Train: {len(df_train)} | Dev: {len(df_dev)} | Test: {len(df_test)}")
print(f"\nTrain Stance:\n{df_train['stance'].value_counts()}")
print(f"\nDev Stance:\n{df_dev['stance'].value_counts()}")
print(f"\nTest Stance:\n{df_test['stance'].value_counts()}")

Train: 1814 | Dev: 200 | Test: 300

Train Stance:
stance
oppose     1139
support     675
Name: count, dtype: int64

Dev Stance:
stance
oppose     132
support     68
Name: count, dtype: int64

Test Stance:
stance
oppose     190
support    110
Name: count, dtype: int64


In [13]:
# Model name
MODEL_NAME = "roberta-base"   # "bert-base-uncased"

In [16]:
#Load tokenized datasets
tokenized_dir = f"tokenized/text/{MODEL_NAME.replace('/', '_')}_maxlen124"

train_dataset_tok = Dataset.load_from_disk(os.path.join(tokenized_dir, "train"))
dev_dataset_tok = Dataset.load_from_disk(os.path.join(tokenized_dir, "dev"))
test_dataset_tok = Dataset.load_from_disk(os.path.join(tokenized_dir, "test"))

print(f"Train dataset loaded with samples: {len(train_dataset_tok)}")
print(f"Dev dataset loaded with samples:: {len(test_dataset_tok)}")
print(f"Test dataset loaded with samples:: {len(dev_dataset_tok)}")

Train dataset loaded with samples: 1814
Dev dataset loaded with samples:: 300
Test dataset loaded with samples:: 200


In [17]:
#Load pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
# Now, we're going to define some metrics to compute
#Classic Metrics: Accuracy, F1, Precision and Recall
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary', pos_label=1, zero_division=0)
    precision = precision_score(labels, preds, average='binary', pos_label=1, zero_division=0)
    recall = recall_score(labels, preds, average='binary', pos_label=1, zero_division=0)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


In [19]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./models/roberta_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none")

In [20]:
#Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tok,
    eval_dataset=dev_dataset_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [21]:
# Fine-tune model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.508800,0.246259,0.880000,0.823529,0.823529,0.823529
2,0.283500,0.278243,0.870000,0.828947,0.750000,0.926471
3,0.191000,0.352921,0.905000,0.863309,0.845070,0.882353
4,0.126700,0.348943,0.915000,0.879433,0.849315,0.911765
5,0.086300,0.394792,0.915000,0.875912,0.869565,0.882353


TrainOutput(global_step=570, training_loss=0.23925904893038566, metrics={'train_runtime': 40.0768, 'train_samples_per_second': 226.315, 'train_steps_per_second': 14.223, 'total_flos': 577960433090400.0, 'train_loss': 0.23925904893038566, 'epoch': 5.0})

In [22]:
# Evaluate
eval_results = trainer.evaluate(test_dataset_tok)
print("\nEvaluation results:", eval_results)



Evaluation results: {'eval_loss': 0.8181412220001221, 'eval_accuracy': 0.8266666666666667, 'eval_f1': 0.8, 'eval_precision': 0.6933333333333334, 'eval_recall': 0.9454545454545454, 'eval_runtime': 0.3102, 'eval_samples_per_second': 967.163, 'eval_steps_per_second': 61.254, 'epoch': 5.0}


In [31]:
example = "I believe stricter gun laws would make our communities safer."

model.eval()

inputs = tokenizer(example, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = outputs.logits.argmax(dim=1).item()
print(pred)

1


In [32]:
# We save fine-tuned model and tokenizer
model.save_pretrained("models/roberta_finetuned")
tokenizer.save_pretrained("models/roberta_finetuned")

('models/roberta_finetuned/tokenizer_config.json',
 'models/roberta_finetuned/special_tokens_map.json',
 'models/roberta_finetuned/vocab.json',
 'models/roberta_finetuned/merges.txt',
 'models/roberta_finetuned/added_tokens.json',
 'models/roberta_finetuned/tokenizer.json')